# Ñu: Lenguaje de Programación en Español



In [ ]:
import os
for f in os.listdir():
    if f.startswith("ñu") or (f.endswith(".py") or f.endswith(".tokens")):
        os.remove(f)

In [ ]:
!pip install -U antlr4-python3-runtime &> /dev/null

In [ ]:
%%writefile ñu.g4
grammar ñu;

root: stat+ EOF;

stat: tipo ID ASIG expr      # Declaracion
    | ID ASIG expr           # Asignacion
    | mostrarStat            # Print
    | ifStat                 # ToIf
    | expr                   # ExprStat
    ;

mostrarStat: MOSTRAR LPAREN expr RPAREN;

tipo: NUM_TIPO | TEXTO_TIPO | BOOL_TIPO | AUTO;

expr: compExpr;

compExpr: addExpr (op=(IGUAL | DIF | MENOR_IGUAL | MAYOR_IGUAL | MENOR | MAYOR) addExpr)? # Comparador
        ;

ifStat : SI expr bloque
        (ELIF expr bloque)*
        (SINO bloque)?
        # Condition
       ;

bloque: LKEY stat* RKEY;

addExpr: addExpr op=(MAS | MENOS) mulExpr # AddSub
        | mulExpr                         # ToMul
        ;

mulExpr: mulExpr MULT powExpr  # Multiplicacion
        | mulExpr DIV powExpr  # Division
        | powExpr              # ToPow
        ;

powExpr: atom ELEVADO powExpr  # Potencia
      | atom                   # ToAtom
      ;

atom
    : LPAREN expr RPAREN     # Parentesis
    | NUM                    # Numero
    | STRING                 # Texto
    | BOOL                   # Booleano
    | ID                     # Variable
    | ingresarExpr           # Input
    | MENOS atom             # Negativo
    ;

ingresarExpr: INGRESAR LPAREN RPAREN;


// === Signos y palabras reservadas ===
SI: 'si';
SINO: 'sino';
ELIF: 'osino';

LKEY: '{';
RKEY: '}';

IGUAL: '==';
DIF: '!=';
MENOR_IGUAL: '<=';
MAYOR_IGUAL: '>=';
MENOR: '<';
MAYOR: '>';

MOSTRAR: 'mostrar';
INGRESAR: 'ingresar';

NUM_TIPO: 'num';
TEXTO_TIPO: 'texto';
BOOL_TIPO: 'bool';
AUTO: 'auto';

BOOL: 'verdadero' | 'falso';
STRING: '"' .*? '"' | '\'' .*? '\'';
NUM : [0-9]+ ('.' [0-9]+)? ;

ID  : [a-zA-ZáéíóúÁÉÍÓÚñÑ_][a-zA-ZáéíóúÁÉÍÓÚñÑ_0-9]* ;

ASIG: '=';

LPAREN: '(';
RPAREN: ')';
MULT: '*';
DIV: '/';
MAS : '+' ;
MENOS : '-' ;
ELEVADO: '**';


WS : [ \t\r\n]+ -> skip ;

Writing ñu.g4


In [ ]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar &> /dev/null

In [ ]:
!java -cp .:antlr-4.13.2-complete.jar org.antlr.v4.Tool ñu.g4 -no-listener -visitor -Dlanguage=Python3

In [ ]:
%%writefile EvalVisitor.py
from antlr4 import *
import operator

oper = {
    '+': operator.add,
    '-': operator.sub,
    '*': operator.mul,
    '/': operator.truediv,
    '**': operator.pow
}

rel = {
    '==': operator.eq,
    '!=': operator.ne,
    '<': operator.lt,
    '>': operator.gt,
    '<=': operator.le,
    '>=': operator.ge
}


if __name__ is not None and "." in __name__:
    from .ñuParser import ñuParser
    from .ñuVisitor import ñuVisitor
else:
    from ñuParser import ñuParser
    from ñuVisitor import ñuVisitor


class EvalVisitor(ñuVisitor):
    def __init__(self):
        self.memory = {}

    def inferir_tipo(self, valor): # -> para auto
        if isinstance(valor, bool):
            return "bool"
        elif isinstance(valor, (int, float)):
            return "num"
        elif isinstance(valor, str):
            return "texto"
        else:
            raise Exception("Tipo no soportado")

    def validar_tipo(self, tipo, valor):
        if tipo == "num":
            return isinstance(valor, (int, float))
        elif tipo == "texto":
            return isinstance(valor, str)
        elif tipo == "bool":
            return isinstance(valor, bool)
        return False

    def visitRoot(self, ctx):
        l = list(ctx.getChildren())

        for i in range(len(l) - 1):  # ignorar EOF
            result = self.visit(l[i])

        return result

    def visitAddSub(self, ctx):
        l = list(ctx.getChildren())

        return oper[l[1].getText()](
            self.visit(l[0]),
            self.visit(l[2])
        )

    def visitMulDiv(self, ctx):
        l = list(ctx.getChildren())

        return oper[l[1].getText()](
            self.visit(l[0]),
            self.visit(l[2])
        )

    def visitPotencia(self, ctx):
        l = list(ctx.getChildren())

        return oper[l[1].getText()](
            self.visit(l[0]),
            self.visit(l[2])
        )

    def visitComparador(self, ctx):
        l = list(ctx.getChildren())

        if len(l) == 1: # -> expresión normal
            return self.visit(l[0])

        return int(
            rel[l[1].getText()](
                self.visit(l[0]),
                self.visit(l[2])
            )
        )


    def visitDeclaracion(self, ctx): # -> declarar por 1ra vez
        l = list(ctx.getChildren())

        tipo = l[0].getText()
        nombre = l[1].getText()
        valor = self.visit(l[3])

        if tipo == 'auto':
            tipo = self.inferir_tipo(valor)

        if not self.validar_tipo(tipo, valor):
            raise Exception(f"Tipo incompatible para '{nombre}'")

        self.memory[nombre] = {
            'tipo': tipo,
            'valor': valor
        }

        return valor

    def visitAsignacion(self, ctx): # -> visitar el valor de otra variable (y = x + 1)
        l = list(ctx.getChildren())

        nombre = l[0].getText()
        valor = self.visit(l[2])

        if nombre not in self.memory:
            raise Exception(f"Variable '{nombre}' no definida")

        tipo = self.memory[nombre]['tipo']

        if not self.validar_tipo(tipo, valor):
            raise Exception(f"Tipo incompatible para '{nombre}'")

        self.memory[nombre]['valor'] = valor
        return valor


    def visitVariable(self, ctx):
        nombre = ctx.getText()

        if nombre in self.memory:
            return self.memory[nombre]['valor']

        raise Exception(f"Variable '{nombre}' no definida")

    def visitNumero(self, ctx):
        texto = ctx.getText()
        return float(texto) if '.' in texto else int(texto)

    def visitTexto(self, ctx):
        texto = ctx.getText()
        return texto[1:-1]

    def visitBooleano(self, ctx):
        return True if ctx.getText() == 'verdadero' else False

    def visitCondition(self, ctx):
        l = list(ctx.getChildren())

        # Si principal (if)
        if self.visit(l[1]) == 1: # -> l[1] es la expresion del "si" (x > 5)
            return self.visit(l[2]) # -> l[2] es el bloque asociado al "si"

        # osino (elif)
        index = 3 # -> es 3 porque los primeros 3 indices (0,1,2) son del "si"
        while index < len(l): # -> para recorrer todos los 'osino' (elifs)
            texto = l[index].getText()
            if texto == 'osino':
                condicion = l[index + 1]  # -> expresion del osino
                bloque = l[index + 2] # ->  # Bloque del osino

                if self.visit(condicion) == 1:
                    return self.visit(bloque)
                index += 3 # avanza 3 pos: osino + expr + bloque (puede ser otro osino o un sino)

            # sino (else)
            elif texto == 'sino':
                return self.visit(l[index + 1]) # -> ya solo retorna pq todo lo anterior no se dio
            else:
                index += 1

    def visitBloque(self, ctx):
        l = list(ctx.getChildren())

        for child in l:
            if child.getText() not in ['{', '}']: # -> ignora los brackets
                result = self.visit(child)

        return result

    def visitPrint(self, ctx):
        print(self.visit(ctx.mostrarStat().expr()))

    def visitIngresarExpr(self, ctx):
        texto = input()

        try:
            return float(texto) if '.' in texto else int(texto)
        except:
            return texto

Writing EvalVisitor.py


## VisitCondition

Explicacion del visitcondition

```
l = [
    token_SI,          # Índice 0: "si"
    ctx_expr_x_5,      # Índice 1: expresión (x > 5)
    ctx_bloque_1,      # Índice 2: bloque { mostrar("Mayor"); }
    token_ELIF_1,      # Índice 3: "osino"
    ctx_expr_x_5_2,    # Índice 4: expresión (x == 5)
    ctx_bloque_2,      # Índice 5: bloque { mostrar("Igual"); }
    token_SINO,        # Índice 6: "sino"
    ctx_bloque_3       # Índice 7: bloque { mostrar("Menor"); }
]
```

In [ ]:
%%writefile ejemplito.ñu
num x = 2
auto y = x + 1
mostrar(y)

Overwriting ejemplito.ñu


In [ ]:
%%writefile ejemplito.ñu
num edad = ingresar()

si edad > 18 {
  mostrar("Mayor de edad")
}
osino edad == 18 {
  mostrar('Es 18 justo')
}
sino {
  mostrar("Es menor de edad")
}

Overwriting ejemplito.ñu


In [ ]:
from antlr4 import *
from ñuLexer import ñuLexer
from ñuParser import ñuParser
from EvalVisitor import EvalVisitor

input_stream = FileStream("ejemplito.ñu")
lexer = ñuLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = ñuParser(token_stream)
tree = parser.root()
# print(tree.toStringTree(recog=parser))

visitor = EvalVisitor()
visitor.visitRoot(tree)

16
Es menor de edad
